In [1]:
import cv2
import numpy as np

# This is the exact data contract agreed upon for Module 2 -> Module 3
mock_blob = {
    "id": 1,
    "bbox": (100, 150, 45, 12),  # (x, y, width, height)
    "contour": np.array([[[100, 150]], [[145, 150]], [[145, 162]], [[100, 162]]]),
    "centroid": (122.5, 156.0),
    "area_px": 540,
    "polarity": "removed"  # Will be 'added' or 'removed'
}

print("Mock blob loaded successfully!")

Mock blob loaded successfully!


In [2]:
def extract_descriptors(blob):
    contour = blob["contour"]
    x, y, w, h = blob["bbox"]
    area = blob["area_px"]
    
    # 1. Perimeter
    perimeter = cv2.arcLength(contour, True)
    
    # 2. Aspect Ratio (Width / Height)
    aspect_ratio = float(w) / h if h != 0 else 0
    
    # 3. Extent (Object Area / Bounding Box Area)
    bounding_box_area = w * h
    extent = float(area) / bounding_box_area if bounding_box_area != 0 else 0
    
    # 4. Solidity (Object Area / Convex Hull Area)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = float(area) / hull_area if hull_area != 0 else 0
    
    # 5. Eccentricity & 6. Hu Moments (derived from image moments)
    moments = cv2.moments(contour)
    hu_moments = cv2.HuMoments(moments).flatten()
    
    # Eccentricity calculation from spatial moments
    if moments['mu20'] + moments['mu02'] != 0:
        eccentricity = ((moments['mu20'] - moments['mu02'])**2 + 4 * moments['mu11']**2)**0.5 / (moments['mu20'] + moments['mu02'])
    else:
        eccentricity = 0

    return {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "hu_moments": hu_moments
    }

# Test the function
descriptors = extract_descriptors(mock_blob)
print("Extracted Descriptors:")
for key, value in descriptors.items():
    print(f"{key}: {value}")

Extracted Descriptors:
area: 540
perimeter: 114.0
aspect_ratio: 3.75
extent: 1.0
solidity: 1.0
eccentricity: 0.8672199170124482
hu_moments: [0.33472222 0.08426119 0.         0.         0.         0.
 0.        ]


In [3]:
def classify_defect(blob, descriptors):
    polarity = blob["polarity"]
    aspect_ratio = descriptors["aspect_ratio"]
    solidity = descriptors["solidity"]
    
    predicted_class = "Unknown"
    
    # STAGE 1: Copper-Removed vs Copper-Added
    if polarity == "removed":
        # STAGE 2 (Removed): Open circuit, Mouse bite, Missing hole
        if aspect_ratio > 3.0:  
            # Highly elongated missing copper is likely a broken trace
            predicted_class = "Open circuit"
        elif solidity > 0.9 and 0.8 < aspect_ratio < 1.2:
            # Solid, roughly square/circular missing copper is a missing hole
            predicted_class = "Missing hole/pin-hole"
        else:
            # Irregular chunks taken out of a trace
            predicted_class = "Mouse bite"
            
    elif polarity == "added":
        # STAGE 2 (Added): Short, Spur, Spurious copper
        if aspect_ratio > 3.0:
            # Elongated added copper connecting traces
            predicted_class = "Short"
        elif solidity > 0.85 and 0.8 < aspect_ratio < 1.2:
            # Solid, isolated islands of copper
            predicted_class = "Spurious copper"
        else:
            # Unwanted protrusion attached to an existing trace
            predicted_class = "Spur"
            
    return predicted_class

# Test the classification
final_class = classify_defect(mock_blob, descriptors)
print(f"Based on the polarity '{mock_blob['polarity']}' and aspect ratio of {descriptors['aspect_ratio']},")
print(f"The system classifies this defect as: {final_class}")

Based on the polarity 'removed' and aspect ratio of 3.75,
The system classifies this defect as: Open circuit
